# 1단계 · 새 VGGFace2 Parent 학습과 검증

같은 RUN_ID를 사용합니다. [실행과 결과 안내](HOW_TO_READ.md)

Parent v3은 batch 128, num_workers 6, prefetch_factor 4로 학습합니다. 매 epoch에 VGGFace2 전체 3,141,890장을 `SEED + epoch`로 다시 섞고, `drop_last=True`로 마지막 2장을 제외합니다. 따라서 24,546배치 × 128장 = 3,141,888장으로 학습합니다. 적용값은 `plan.json`의 `recipe.drop_last`에서 확인합니다.

처음 두 epoch의 전체 step에 걸쳐 LR을 0.0025에서 0.025까지 선형으로 올리고 gradient L2 norm을 5.0으로 제한합니다. Conv·Linear·BN2d·PReLU 가중치의 WD는 1e-4이며, 출력 BN·head·bias의 WD는 0입니다. 매 epoch의 health gate는 이전 최저 loss보다 3 초과 상승(epoch 2부터), PReLU 절댓값 10 초과, BN running variance 100,000 초과 중 하나가 발생하면 체크포인트와 진단치를 저장하고 학습을 중단합니다. LFW는 중앙 144×144 ROI로 PFLD 5점을 구해 원본에서 112×112로 정렬합니다.

In [ ]:
# cell 1 : 설정 및 경로 확인
import json
import os
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "fine-tune/modules").is_dir())
sys.path.insert(0, str(repo / "fine-tune"))
from modules.clean_training import final_artifact, recipe, run_directory, run_stage
from modules.clean_campaign import model_dir, origin_matrix, parent_validation, read_result, run_full, run_probe, validate_choices

DATA_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets"))
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models"))
RUN_ID = "clean_mfn1024_s8972_v3"
SEED = 8972
EXECUTE = False
RESUME = False
KEEP_RAW_RESULTS = False
VALIDATE_PARENT_ONLY = False
run = run_directory(MODEL_ROOT, RUN_ID)
print(run / "parent")
display(recipe("parent"))

In [ ]:
# cell 2 : 실행
if EXECUTE:
    if not VALIDATE_PARENT_ONLY and not (run / "parent/COMPLETED").is_file():
        run_stage(DATA_ROOT, run, "parent", seed=SEED, keep_raw_results=KEEP_RAW_RESULTS, resume=RESUME)
    parent_result = parent_validation(DATA_ROOT, run, seed=SEED, keep_raw_results=KEEP_RAW_RESULTS)

In [ ]:
# cell 3 : 결과 확인
if EXECUTE:
    display(pd.DataFrame(parent_result["comparison"]))
elif (run / "parent_validation/COMPLETED").is_file():
    display(pd.DataFrame(read_result(run / "parent_validation")["comparison"]))
else:
    progress_path = run / "parent/progress.json"
    if progress_path.is_file():
        epochs = json.loads(progress_path.read_bytes())["epochs"]
        fields = ["epoch", "loss", "ending_lrs", "grad_norm_clipped_max", "max_abs_prelu",
                  "max_bn_running_var", "max_linear_row_norm", "head_direction_norm", "health_gate_failure"]
        display(pd.DataFrame(epochs).reindex(columns=fields))
        display(pd.DataFrame([
            dict(epoch=r["epoch"], lfw=100*r["zero_shot"]["metrics"]["lfw"]["accuracy"],
                 scface=100*r["zero_shot"]["metrics"]["scface"]["rank1"],
                 tinyface=100*r["zero_shot"]["metrics"]["tinyface"]["rank1"],
                 seconds=r["zero_shot"]["seconds"])
            for r in epochs if "zero_shot" in r
        ]))
    print("완료 후 단독 검증:", run / "parent_validation/summary.json")